# 04 · Orchestrate — 04 Plan, act, reflect, retry: the loop closed

**The three previous notebooks each built one decision. Notebook 01 decides whether to retrieve, notebook 02 decides where from, notebook 03 decides whether to try again. This notebook runs them as one cycle, where reflecting on a result feeds back into planning the next attempt — and the plan a second cycle acts on is not the plan the first cycle used.**

Planning exists in pieces across this lab; so does reflection. What does not
exist anywhere is the two connected, so that a reflection changes a
subsequent plan. That connection is the only genuinely new thing here. The
plan step is notebook 01's router plus notebook 02's ordering; the reflect
step is notebook 03's grounding verdict. What is new is the arrow back from
reflect to plan, and the cap that keeps that arrow from becoming a circle
with no exit.

```
  plan ──▶ act ──▶ reflect ──┬──▶ accept   (verdict is not weak)
    ▲                        ├──▶ abstain  (cap reached, still weak)
    └──── revise ────────────┘             (weak, attempts left)
```

## What this notebook demonstrates

| Name | What it does | Example |
|---|---|---|
| `plan` | Turns a question into a plan: retrieve or not, source order, query to use | `plan("how do mitochondria produce ATP?")` |
| `act` | Executes one plan against the stores, merging into evidence already gathered | `act(p, evidence)` |
| `reflect` | Grounding verdict on the observation, plus one of accept / revise / abstain | `reflect(obs, cycle, cap)` |
| `revise` | Builds the *next* plan from the reflection, not from the question | `revise(p, cycle)` |
| `run_loop` | The closed cycle, capped, with every cycle logged | `run_loop(q, max_cycles=2)` |
| `ABSTENTION` | The output when the cap is reached and the verdict is still weak | `ABSTENTION` |

In [ ]:
import sys
from pathlib import Path

_root = Path.cwd().resolve()
for _ in range(6):
    if (_root / "nbio.py").is_file():
        break
    _root = _root.parent
else:
    raise RuntimeError("could not locate nbio.py above the current directory")
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import nbio
nbio.bootstrap()

## Step 1 — the two stores and the known-context facts, carried in from 01 and 02

Nothing new in this cell: the offline hash embedding, two small document
sets tagged by domain, a counter on every lookup, and the session facts
that make a lookup unnecessary. All of it is the previous notebooks'
material, kept here so this notebook runs standalone like every other one
in this repo.

In [ ]:
import hashlib
import math

RETRIEVAL_CALLS = []


def hash_embed(text: str, dim: int = 384) -> list[float]:
    vec = [0.0] * dim
    for tok in (text or "").lower().split():
        h = int(hashlib.sha256(tok.encode("utf-8")).hexdigest(), 16)
        vec[h % dim] += 1.0 if (h >> 8) & 1 else -1.0
    norm = math.sqrt(sum(v * v for v in vec)) or 1.0
    return [v / norm for v in vec]


def cosine(a: list[float], b: list[float]) -> float:
    dot = sum(x * y for x, y in zip(a, b))
    na = math.sqrt(sum(x * x for x in a)) or 1.0
    nb = math.sqrt(sum(x * x for x in b)) or 1.0
    return dot / (na * nb)


SOURCES = {
    "course_store": {"domains": {"course"}, "docs": [
        {"chunk_id": "bio201::c0", "title": "Cellular respiration",
         "text": "Mitochondria perform oxidative phosphorylation, converting nutrients into ATP."},
        {"chunk_id": "bio201::c1", "title": "Photosynthesis",
         "text": "Photosynthesis converts light energy into chemical energy stored in glucose."},
    ]},
    "clinical_index": {"domains": {"clinical"}, "docs": [
        {"chunk_id": "burns::c0", "title": "Pediatric burn management",
         "text": "Early excision and grafting reduces mortality in major pediatric burns."},
        {"chunk_id": "burns::c1", "title": "Grafting techniques",
         "text": "Split-thickness skin grafts are preferred for large surface areas."},
    ]},
}
for _s in SOURCES.values():
    for _d in _s["docs"]:
        _d["embedding"] = hash_embed(_d["text"])

KNOWN_CONTEXT = {
    "course_id": {"value": "bio201", "triggers": {"course", "id", "which class", "what class"}},
}


def search(source_name: str, query: str, top_k: int = 2) -> list[dict]:
    RETRIEVAL_CALLS.append((source_name, query))
    qvec = hash_embed(query)
    hits = [{"chunk_id": d["chunk_id"], "title": d["title"], "text": d["text"],
             "source": source_name, "score": cosine(qvec, d["embedding"])}
            for d in SOURCES[source_name]["docs"]]
    hits.sort(key=lambda h: -h["score"])
    return hits[:top_k]


print("sources:", {k: len(v["docs"]) for k, v in SOURCES.items()})
print("known context keys:", list(KNOWN_CONTEXT))

## Step 2 — the pieces this loop reuses, in their smallest form

The router from notebook 01, the domain ordering from notebook 02, the
generator stand-in and grounding verdict from notebook 03. Each is reduced
to what the loop needs — the full versions, with their cue lists, their
cost/latency policy and their citation checks, are in those notebooks.

`synthesize` keeps the behaviour that makes the whole stage testable
offline: when nothing retrieved clears `SUPPORT_FLOOR`, it writes a
confident sentence that is in no document. `04-benchmarks` calls that
failure *answered-without-evidence*; here it is what the reflect step has
to catch.

In [ ]:
import re

DOMAIN_CUES = {
    "course": [re.compile(p) for p in [r"\bmitochondri", r"\batp\b", r"\bphotosynthes", r"\bcell\b"]],
    "clinical": [re.compile(p) for p in [r"\bburn", r"\bgraft", r"\bmortality\b", r"\bsepsis\b",
                                         r"\bvasopressor\b", r"\bpatient", r"\btrial\b"]],
}

SUPPORT_FLOOR = 0.15
UNSUPPORTED_SENTENCE = "The retrieved sources state this directly and without qualification."


def context_hit(question: str) -> str | None:
    q = (question or "").lower()
    for key, entry in KNOWN_CONTEXT.items():
        if len([t for t in entry["triggers"] if t in q]) >= 2:
            return key
    return None


def needs_retrieval(question: str) -> bool:
    """Notebook 01's router, reduced to the branch this loop exercises: an
    already-known fact needs no lookup, and everything else defaults to
    retrieving, which is the cheaper of the two mistakes. The cue lists that
    also skip greetings and arithmetic are in 01-should-i-retrieve.ipynb."""
    return context_hit(question) is None


def source_order(question: str) -> list[str]:
    tags = {d for d, pats in DOMAIN_CUES.items() if any(p.search(question.lower()) for p in pats)}
    return sorted(SOURCES, key=lambda n: (0 if SOURCES[n]["domains"] & tags else 1, n))


def merge(previous: list[dict], new_hits: list[dict]) -> list[dict]:
    merged = {r["chunk_id"]: r for r in previous}
    for r in new_hits:
        prev = merged.get(r["chunk_id"])
        if prev is None or r["score"] > prev["score"]:
            merged[r["chunk_id"]] = r
    return sorted(merged.values(), key=lambda x: -x["score"])


def synthesize(results: list[dict]) -> str:
    if results and results[0]["score"] >= SUPPORT_FLOOR:
        return f'\U0001F4C4 "{results[0]["text"]}"'
    return f'\U0001F4C4 "{UNSUPPORTED_SENTENCE}"'


def norm(s: str) -> str:
    return " ".join((s or "").lower().split())


def grounding_verdict(answer_text: str, papers: list[dict]) -> dict:
    """grounded True/False/None on the deterministic signal, as 01-tools/05-gate produces it."""
    if not papers or not (answer_text or "").strip():
        return {"grounded": False, "score": 0.0, "unsupported": [], "method": "precondition"}
    ctx = "\n".join(norm(p.get("title", "")) + " " + norm(p["text"]) for p in papers)
    quotes = re.findall(r'"([^"\n]{12,400})"', answer_text)
    unsupported = [q for q in quotes if norm(q) not in ctx]
    if unsupported:
        return {"grounded": False, "score": 0.5, "unsupported": unsupported[:3], "method": "deterministic"}
    return {"grounded": None, "score": 1.0, "unsupported": [], "method": "deterministic"}


def is_weak(verdict: dict) -> bool:
    return verdict["grounded"] is False or (verdict["grounded"] is None and verdict["score"] < 0.7)


print("router on a course question :", needs_retrieval("how do mitochondria produce ATP?"))
print("router on a known fact      :", needs_retrieval("what course id is this session?"))
print("order for a clinical question:", source_order("does early excision reduce mortality in burns?"))

assert needs_retrieval("what course id is this session?") is False
assert source_order("does early excision reduce mortality in burns?")[0] == "clinical_index"

## Step 3 — `plan`: a question becomes an explicit, inspectable plan

The plan is data, not control flow. That is the difference between a
pipeline with branches in it and a loop that can revise itself: a branch
taken inside a function cannot be revised afterwards, because there is
nothing left to revise. A plan that is a dict can be printed, diffed
against the next one, and changed by the reflect step.

Each plan carries the cycle it belongs to and the reason it exists, so the
log in Step 6 can show *why* cycle 2's plan differed from cycle 1's.

In [ ]:
def plan(question: str, cycle: int = 1, query: str | None = None, reason: str = "initial plan") -> dict:
    return {
        "cycle": cycle,
        "question": question,
        "needs_retrieval": needs_retrieval(question),
        "context_key": context_hit(question),
        "source_order": source_order(question) if needs_retrieval(question) else [],
        "query": query or question,
        "reason": reason,
    }


p_course = plan("how do mitochondria produce ATP?")
p_known = plan("what course id is this session?")

nbio.show_json(p_course)
print()
nbio.show_json(p_known)

assert p_course["needs_retrieval"] is True and p_course["source_order"][0] == "course_store"
assert p_known["needs_retrieval"] is False and p_known["source_order"] == []

## Step 4 — `act`: execute the plan, accumulating evidence across cycles

`act` takes the evidence gathered so far and merges this cycle's hits into
it, using notebook 03's max-score-per-chunk rule. Cycles accumulate rather
than replace, so a later cycle can only ever improve the evidence set — a
revision that retrieves nothing useful leaves the previous cycle's best
hits intact.

The no-retrieval branch answers straight from `KNOWN_CONTEXT` and touches
no store at all, which the retrieval counter confirms in Step 6.

In [ ]:
def act(current_plan: dict, evidence: list[dict]) -> dict:
    if not current_plan["needs_retrieval"]:
        key = current_plan["context_key"]
        value = KNOWN_CONTEXT[key]["value"]
        return {"answer": f"{key} = {value}", "evidence": evidence, "used_retrieval": False,
                "top_score": None}

    for source_name in current_plan["source_order"]:
        evidence = merge(evidence, search(source_name, current_plan["query"]))
        if evidence and evidence[0]["score"] >= SUPPORT_FLOOR:
            break  # good enough to stop probing sources this cycle

    return {"answer": synthesize(evidence), "evidence": evidence, "used_retrieval": True,
            "top_score": round(evidence[0]["score"], 3) if evidence else 0.0}


RETRIEVAL_CALLS.clear()
obs_known = act(p_known, [])
print("known-fact plan ->", obs_known["answer"], "| retrievals:", len(RETRIEVAL_CALLS))

RETRIEVAL_CALLS.clear()
obs_course = act(p_course, [])
print("course plan     ->", obs_course["answer"], "| retrievals:", len(RETRIEVAL_CALLS))

assert obs_known["used_retrieval"] is False
assert len(RETRIEVAL_CALLS) >= 1 and obs_course["used_retrieval"] is True

## Step 5 — `reflect` and `revise`: the arrow back to planning

`reflect` reads the observation and returns the grounding verdict plus one
of three decisions. Only one of them continues the loop.

- **accept** — the verdict is not weak; the loop is done.
- **abstain** — the verdict is weak and the cap is reached, or there is no
  revision left to make. The answer is replaced with an abstention.
- **revise** — the verdict is weak and there is another cycle available.

`revise` is what makes this a loop rather than a sequence: it builds the
next plan from the *reflection*, drawing the next query from a
reformulation pool. The question never changes; the plan does.

The abstention string starts with `INSUFFICIENT EVIDENCE`, which is
exactly the prefix `01-tools/05-gate`'s `check_grounding` short-circuits
on. Step 8 checks that, so the loop's failure output is one the existing
gate already recognises rather than a new convention invented here.

In [ ]:
ABSTENTION = ("INSUFFICIENT EVIDENCE: the retrieved material did not support an answer "
              "after every planned attempt.")

REFORMULATIONS = {
    "bio": ["What is the role of mitochondria in oxidative phosphorylation?",
            "mitochondria ATP production",
            "oxidative phosphorylation nutrients"],
    "sepsis": ["sepsis vasopressor timing 2026",
               "norepinephrine timing septic shock trial",
               "early vasopressor sepsis outcome"],
}


def reflect(observation: dict, cycle: int, max_cycles: int, pool: list[str]) -> dict:
    verdict = grounding_verdict(observation["answer"], observation["evidence"]) \
        if observation["used_retrieval"] else {"grounded": None, "score": 1.0, "unsupported": [],
                                               "method": "no-retrieval"}
    weak = is_weak(verdict)
    has_revision = cycle - 1 < len(pool)
    if not weak:
        decision, why = "accept", "verdict is not weak"
    elif cycle >= max_cycles:
        decision, why = "abstain", f"weak, and cycle {cycle} of {max_cycles} is the cap"
    elif not has_revision:
        decision, why = "abstain", "weak, and no reformulation left to try"
    else:
        decision, why = "revise", f"weak -> next query {pool[cycle - 1]!r}"
    return {"verdict": verdict, "decision": decision, "why": why}


def revise(previous_plan: dict, cycle: int, pool: list[str]) -> dict:
    return plan(previous_plan["question"], cycle=cycle + 1, query=pool[cycle - 1],
                reason=f"cycle {cycle}'s verdict was weak")


demo_reflection = reflect(obs_course, cycle=1, max_cycles=2, pool=REFORMULATIONS["bio"])
print(demo_reflection["decision"], "—", demo_reflection["why"])

assert reflect({"answer": f'\U0001F4C4 "{UNSUPPORTED_SENTENCE}"', "evidence": obs_course["evidence"],
                "used_retrieval": True}, 1, 2, REFORMULATIONS["bio"])["decision"] == "revise"
assert reflect({"answer": f'\U0001F4C4 "{UNSUPPORTED_SENTENCE}"', "evidence": obs_course["evidence"],
                "used_retrieval": True}, 2, 2, REFORMULATIONS["bio"])["decision"] == "abstain"

## Step 6 — `run_loop`: the cycle closed, and capped

`max_cycles` is a required argument with a small default, and the loop
returns from inside it — there is no `while` here whose exit depends on a
verdict ever improving. That is deliberate: notebook 03 showed what a
verdict that never improves does to an unbounded retry, and the same
discipline applies to the full cycle, where each iteration costs a
generation call as well as a retrieval.

Every cycle is logged with its plan, what it retrieved, the verdict, and
the decision — which is what lets the tables below show a plan changing
because of a reflection rather than merely asserting that it did.

In [ ]:
MODEL_ID = "llama-3.1-8b-instant"


def run_loop(question: str, pool: list[str], max_cycles: int = 2, meter=None) -> dict:
    assert max_cycles >= 1, "at least one cycle, and a finite number of them"
    current, evidence, log = plan(question), [], []

    for cycle in range(1, max_cycles + 1):
        observation = act(current, evidence)
        evidence = observation["evidence"]
        reflection = reflect(observation, cycle, max_cycles, pool)
        if meter is not None:
            # No paid call on the offline path, so zero tokens — the entry still
            # puts this cycle's generation step through the ceiling. With a key
            # set, the real usage from that call is recorded here instead.
            meter.record(MODEL_ID, 0, 0)

        log.append({"cycle": cycle, "query": current["query"], "plan_reason": current["reason"],
                    "sources": ",".join(current["source_order"]) or "(none)",
                    "top_score": observation["top_score"],
                    "grounded": reflection["verdict"]["grounded"],
                    "decision": reflection["decision"], "why": reflection["why"]})

        if reflection["decision"] == "accept":
            return {"answer": observation["answer"], "outcome": "accepted", "cycles": cycle,
                    "verdict": reflection["verdict"], "evidence": evidence, "log": log}
        if reflection["decision"] == "abstain":
            return {"answer": ABSTENTION, "outcome": "abstained", "cycles": cycle,
                    "verdict": reflection["verdict"], "evidence": evidence, "log": log}
        current = revise(current, cycle, pool)

    raise AssertionError("unreachable: every cycle ends in accept, abstain, or revise-with-a-cycle-left")


def show_loop(run: dict) -> None:
    nbio.table([(r["cycle"], r["query"][:40], r["sources"][:24], r["top_score"], str(r["grounded"]),
                 r["decision"], r["why"][:46]) for r in run["log"]],
               ("#", "query", "sources", "top", "grounded", "decision", "why"))


RETRIEVAL_CALLS.clear()
with nbio.cost_meter(budget_usd=0.50) as meter:
    bio_run = run_loop("what is the powerhouse of the cell", REFORMULATIONS["bio"], max_cycles=2, meter=meter)
    bio_metered = meter.calls
    print(meter.report())

print()
show_loop(bio_run)
print(f"\noutcome: {bio_run['outcome']} after {bio_run['cycles']} cycle(s)")
print(f"answer  : {bio_run['answer']}")

assert bio_run["cycles"] == 2, "cycle 1's verdict was weak, so the loop planned a second cycle"
assert bio_run["outcome"] == "accepted"
assert bio_run["log"][0]["query"] != bio_run["log"][1]["query"], (
    "the second cycle acted on a different plan — that is the arrow from reflect back to plan"
)
assert bio_run["log"][1]["plan_reason"] == "cycle 1's verdict was weak", (
    "and the new plan records the reflection that caused it"
)
assert bio_metered == bio_run["cycles"]

## Step 6b — the cycle that never starts: a question answered from what is already known

The loop's cheapest path. The plan says no retrieval, `act` answers from
`KNOWN_CONTEXT`, the reflection has no grounding problem to find, and the
loop accepts on cycle one having touched no store at all.

A plan-act-reflect loop that always retrieves is a pipeline with extra
vocabulary. The retrieval counter reading zero is what makes the planning
step load-bearing.

In [ ]:
RETRIEVAL_CALLS.clear()
with nbio.cost_meter(budget_usd=0.50) as meter:
    known_run = run_loop("what course id is this session?", REFORMULATIONS["bio"], max_cycles=2, meter=meter)

show_loop(known_run)
print(f"\nanswer: {known_run['answer']}  |  retrievals made: {len(RETRIEVAL_CALLS)}")

assert known_run["cycles"] == 1 and known_run["outcome"] == "accepted"
assert len(RETRIEVAL_CALLS) == 0, "the planning step must be able to prevent retrieval entirely"

## Step 7 — the cap, proven on a question no store can answer

Nothing here is rigged. The question asks about material that exists in
neither store, so every cycle retrieves the same near-zero scores, every
generation over-claims, and every verdict comes back `grounded: False`.
The verdict genuinely never improves — and the only thing that ends the
loop is the cap.

Run at two different caps, as in notebook 03, because one run stopping at
2 could always have been the data. Two caps producing two different cycle
counts, on identical inputs, can only be the cap. The asserts also confirm
the reformulation pool still had spare entries at the moment each run
stopped, so the pool was not what ran out.

In [ ]:
SEPSIS_Q = "What did the 2026 sepsis trial conclude about vasopressor timing?"

cap_runs = {}
for cap in (2, 3):
    RETRIEVAL_CALLS.clear()
    with nbio.cost_meter(budget_usd=0.50) as meter:
        r = run_loop(SEPSIS_Q, REFORMULATIONS["sepsis"], max_cycles=cap, meter=meter)
        r["metered"] = meter.calls
    r["retrievals"] = len(RETRIEVAL_CALLS)
    cap_runs[cap] = r
    print(f"--- cap = {cap} ---")
    show_loop(r)
    print()

nbio.table([(cap, r["cycles"], r["retrievals"], r["metered"], r["outcome"]) for cap, r in cap_runs.items()],
           ("cap", "cycles run", "retrievals", "metered calls", "outcome"))

for cap, r in cap_runs.items():
    assert r["cycles"] == cap, f"cap {cap} must produce exactly {cap} cycles"
    assert r["outcome"] == "abstained", "still weak at the cap, so it abstains rather than answering"
    assert r["verdict"]["grounded"] is False
    assert r["metered"] == cap
    assert len(REFORMULATIONS["sepsis"]) >= cap, "spare reformulations remained — the pool was not the limit"
    assert all((row["top_score"] or 0.0) < SUPPORT_FLOOR for row in r["log"]), (
        "every cycle genuinely retrieved nothing above the support floor"
    )

assert cap_runs[2]["cycles"] != cap_runs[3]["cycles"], (
    "identical inputs, different caps, different cycle counts — the cap is what stopped it"
)
print()
print(f"confirmed: the loop stopped at its cap in both runs and abstained rather than answering")

## Step 8 — the abstention is the one the gate already recognises

`01-tools/05-gate/01-grounding-check.ipynb` short-circuits before any
check when an answer begins with `INSUFFICIENT EVIDENCE` — it treats that
as the system correctly declining, not as an ungrounded answer. This
loop's give-up output uses that exact prefix, so a downstream gate reads
it correctly with no new convention to learn.

The check below is the gate's own precondition, reimplemented in the two
lines it actually is, and run against this loop's abstention.

In [ ]:
def grounding_precondition(answer_text: str, papers: list[dict]) -> bool:
    """True if a grounding check should proceed at all — 05-gate's first lines."""
    text = (answer_text or "").strip()
    return bool(text) and not text.startswith("INSUFFICIENT EVIDENCE") and bool(papers)


abstained = cap_runs[2]
print("abstention:", abstained["answer"])
print("gate would check it? ", grounding_precondition(abstained["answer"], abstained["evidence"]))
print("gate would check the accepted bio answer?", grounding_precondition(bio_run["answer"], bio_run["evidence"]))

assert grounding_precondition(abstained["answer"], abstained["evidence"]) is False, (
    "the gate recognises this loop's abstention and declines to grade it as an answer"
)
assert grounding_precondition(bio_run["answer"], bio_run["evidence"]) is True
print()
print("confirmed: the loop's give-up output is the abstention 01-tools/05-gate already knows how to read")

## Step 9 — the three outcomes together

Every path this loop can take, in one table: accepted after a revision,
accepted with no retrieval at all, and abstained at the cap. The `cycles`
column is the only number in the stage that reflects a decision made
*after* seeing evidence.

In [ ]:
nbio.table(
    [("informal bio question", bio_run["outcome"], bio_run["cycles"], str(bio_run["verdict"]["grounded"])),
     ("known session fact", known_run["outcome"], known_run["cycles"], str(known_run["verdict"]["grounded"])),
     ("unanswerable question (cap 2)", cap_runs[2]["outcome"], cap_runs[2]["cycles"],
      str(cap_runs[2]["verdict"]["grounded"]))],
    ("case", "outcome", "cycles", "grounded"),
)

outcomes = {bio_run["outcome"], known_run["outcome"], cap_runs[2]["outcome"]}
assert outcomes == {"accepted", "abstained"}, "both terminal outcomes are reachable"
assert {bio_run["cycles"], known_run["cycles"], cap_runs[2]["cycles"]} == {1, 2}, (
    "the number of cycles is decided by the evidence, not fixed in advance"
)
print()
print("confirmed: plan -> act -> reflect -> revise runs as one cycle, and the cycle count is not fixed in advance")

## What this does NOT prove

Stated plainly, because a loop that improves a result on an example built
to be improved is the easiest thing in this repo to overclaim about.

- **A synthetic example is not evidence of real improvement.** Step 6's
  question improves because a hand-written reformulation happens to use
  the store's own vocabulary, in a two-chunk store, with a generator
  stand-in that over-claims on cue. Every part of that was chosen to make
  the mechanism visible. Whether retrying on a weak verdict improves
  answers on *real* questions is an empirical question this notebook does
  not touch — it needs `04-benchmarks/clinical-retrieval/`, run across the
  question set, with and without the loop. That is separate work and it
  has not been done.
- **Step 7's abstention is the right behaviour, not a good outcome.** The
  loop declines to answer a question it cannot support. It does not find
  the answer, and nothing here suggests more cycles would have.
- **Nothing measures whether the loop makes things worse.** A revision can
  pull in a higher-scoring chunk that is less relevant. This stage has no
  instrument for that, either.

## What did not come across

- **No model plans anything.** `plan` and `revise` are rules and a
  hand-written reformulation pool. A model-written plan — the usual
  meaning of "planning agent" — is follow-on work, and would sit behind
  the same key-optional pattern `01-should-i-retrieve.ipynb` Step 7 uses.
- **Single agent only.** One loop, one decision-maker. A supervisor
  routing to specialist agents is explicitly out of scope for this stage.
- **The reflect step is the deterministic grounding pass only.** The LLM
  judge, orphan-citation checks and bibliographic guard live in
  `01-tools/05-gate/01-grounding-check.ipynb` and are not duplicated here.
- **No state survives the loop.** Nothing is remembered between questions;
  each `run_loop` call starts with empty evidence. Carrying state between
  questions is `01-modules/02-memory`'s stage; nothing in this loop reads
  or writes it.